In [2]:
import pandas as pd
import numpy as np

df_master = pd.read_csv('../data/processed/master_features.csv')
print(df_master.shape)

(307511, 149)


In [5]:
correlations = df_master.select_dtypes(include=[np.number]).corr()['TARGET'].sort_values()

print("TOP 15 NEGATIF (makin tinggi nilai, makin AMAN):")
print(correlations.head(15))

print("\nTOP 15 POSITIF (makin tinggi nilai, makin BERISIKO):")
print(correlations.tail(16))

TOP 15 NEGATIF (makin tinggi nilai, makin AMAN):
EXT_SOURCE_2                 -0.160295
EXT_SOURCE_3                 -0.155892
EXT_SOURCE_1                 -0.098887
AGE_YEARS                    -0.078239
EMPLOYED_YEARS               -0.063368
PREV_RATIO_APPROVED          -0.041736
HAS_PROPERTY_INFO            -0.041392
AMT_GOODS_PRICE              -0.039623
BUREAU_COUNT_CLOSED          -0.037233
REGION_POPULATION_RELATIVE   -0.037227
AMT_CREDIT                   -0.030369
POS_COUNT                    -0.029678
FLAG_DOCUMENT_6              -0.028602
PREV_MEAN_AMT_ANNUITY        -0.026242
HOUR_APPR_PROCESS_START      -0.024166
Name: TARGET, dtype: float64

TOP 15 POSITIF (makin tinggi nilai, makin BERISIKO):
REG_CITY_NOT_WORK_CITY                0.050994
DAYS_ID_PUBLISH                       0.051457
DAYS_LAST_PHONE_CHANGE                0.055218
REGION_RATING_CLIENT                  0.058899
REGION_RATING_CLIENT_W_CITY           0.060893
ANOMALY_SCORE                         0.061664
I

In [6]:
feature_cols = [c for c in df_master.select_dtypes(include=[np.number]).columns 
                 if c not in ['TARGET', 'SK_ID_CURR']]

corr_matrix = df_master[feature_cols].corr().abs()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [(col, row, upper.loc[row, col]) 
                    for col in upper.columns 
                    for row in upper.index 
                    if upper.loc[row, col] > 0.9]

high_corr_df = pd.DataFrame(high_corr_pairs, columns=['Feature_1', 'Feature_2', 'Correlation'])
high_corr_df = high_corr_df.sort_values('Correlation', ascending=False)
print(high_corr_df.shape)
print(high_corr_df.head(30))

(14, 3)
                      Feature_1                  Feature_2  Correlation
3                     AGE_YEARS                 DAYS_BIRTH     1.000000
4                EMPLOYED_YEARS              DAYS_EMPLOYED     1.000000
2      OBS_60_CNT_SOCIAL_CIRCLE   OBS_30_CNT_SOCIAL_CIRCLE     0.998491
0               AMT_GOODS_PRICE                 AMT_CREDIT     0.986734
9         INST_MEAN_AMT_PAYMENT   INST_MEAN_AMT_INSTALMENT     0.979679
6          PREV_MEAN_AMT_CREDIT  PREV_MEAN_AMT_APPLICATION     0.977106
11           POS_MAX_SK_DPD_DEF        POS_MEAN_SK_DPD_DEF     0.964838
12                CC_MAX_SK_DPD             CC_MEAN_SK_DPD     0.960642
1   REGION_RATING_CLIENT_W_CITY       REGION_RATING_CLIENT     0.950842
13                 CC_RATIO_DPD            CC_SUM_FLAG_DPD     0.944662
5           BUREAU_COUNT_CLOSED         BUREAU_COUNT_LOANS     0.932987
10               POS_MAX_SK_DPD            POS_MEAN_SK_DPD     0.924367
7       PREV_MEAN_DAYS_LAST_DUE   PREV_MEAN_DAYS_FIRST_D

In [7]:
from sklearn.model_selection import train_test_split

X = df_master.drop(columns=['TARGET'])
y = df_master['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train TARGET rate:", y_train.mean())
print("Test TARGET rate:", y_test.mean())

Train shape: (246008, 148)
Test shape: (61503, 148)
Train TARGET rate: 0.08072908198107379
Test TARGET rate: 0.08072776937710356


In [8]:
# Kolom yang perlu di-drop: FLAG_EMPLOYED_LONGER_THAN_POSSIBLE (zero variance)
# + salah satu dari tiap pasangan high-correlation

cols_to_drop = [
    'FLAG_EMPLOYED_LONGER_THAN_POSSIBLE',  # zero variance
    
    'DAYS_BIRTH',                    # keep AGE_YEARS (lebih interpretable)
    'DAYS_EMPLOYED',                 # keep EMPLOYED_YEARS
    'OBS_60_CNT_SOCIAL_CIRCLE',      # keep OBS_30 (window lebih pendek/relevan)
    'AMT_GOODS_PRICE',               # keep AMT_CREDIT (lebih relevan ke risk)
    'INST_MEAN_AMT_PAYMENT',         # keep INST_MEAN_AMT_INSTALMENT
    'PREV_MEAN_AMT_APPLICATION',     # keep PREV_MEAN_AMT_CREDIT
    'POS_MEAN_SK_DPD_DEF',           # keep POS_MAX_SK_DPD_DEF (max lebih tajam nangkep risiko)
    'CC_MEAN_SK_DPD',                # keep CC_MAX_SK_DPD
    'REGION_RATING_CLIENT',          # keep _W_CITY (lebih granular)
    'CC_SUM_FLAG_DPD',               # keep CC_RATIO_DPD (rasio > raw count, sesuai pola kita)
    'BUREAU_COUNT_CLOSED',           # keep BUREAU_COUNT_LOANS
    'POS_MEAN_SK_DPD',               # keep POS_MAX_SK_DPD
    'PREV_MEAN_DAYS_FIRST_DUE',      # keep PREV_MEAN_DAYS_LAST_DUE
    'INST_SUM_FLAG_LATE',            # keep INST_SUM_FLAG_SHORTFALL
]

X_train_clean = X_train.drop(columns=cols_to_drop)
X_test_clean = X_test.drop(columns=cols_to_drop)

print("Before:", X_train.shape)
print("After:", X_train_clean.shape)

Before: (246008, 148)
After: (246008, 133)


In [9]:
# Simpan SK_ID_CURR terpisah dulu (buat referensi/tracking, bukan buat fitur model)
train_ids = X_train_clean['SK_ID_CURR']
test_ids = X_test_clean['SK_ID_CURR']

X_train_final = X_train_clean.drop(columns=['SK_ID_CURR'])
X_test_final = X_test_clean.drop(columns=['SK_ID_CURR'])

print(X_train_final.shape)
print(X_test_final.shape)

(246008, 132)
(61503, 132)


In [10]:
X_train_final.to_csv('../data/processed/X_train.csv', index=False)
X_test_final.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
train_ids.to_csv('../data/processed/train_ids.csv', index=False)
test_ids.to_csv('../data/processed/test_ids.csv', index=False)

print("Saved!")

Saved!
